# 06 — 方法模式：运行电化学实验

**方法模式**是运行完整电化学实验（LSV、CV、EIS、计时安培法等）的标准工作流。你在 IviumSoft 中加载方法文件，可选择从 Python 修改其参数，启动方法，监控进度，并获取测量数据。

### 工作流概览

```
load_method()          ← 将 .imf 方法文件加载到 IviumSoft
    ↓
set_method_parameter() ← （可选）以编程方式修改参数
    ↓
start_method()         ← 启动测量
    ↓
get_available_data_points_number()  ← 在循环中轮询进度
    ↓
get_data_point(index)  ← 在运行期间或结束后读取数据
    ↓
save_data()            ← 将结果保存到 .idf 文件
```

### 前提条件
- IviumSoft 已安装、正在运行、设备已连接
- 在 IviumSoft 中创建的方法文件（`.imf`）— 内置示例见 `C:/IviumStat/AppMethods/`
- 错误处理参考：`01_getting_started.ipynb` — 第 4 节

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

## 1. 加载方法文件

`load_method()` 将 `.imf` 方法文件加载到 IviumSoft 而不启动它。这可让你在运行前检查或修改参数。

> 请将 `METHOD_PATH` 更新为你系统上方法文件的路径。

In [ ]:
METHOD_PATH = r"C:/IviumStat/datafiles/TEST1.imf"  # 请更新此路径

Pyvium.open_driver()
Pyvium.load_method(METHOD_PATH)
print(f"已加载: {METHOD_PATH}")

## 2. 修改方法参数

`set_method_parameter(name, value)` 更改已加载方法的单个参数。`name` 和 `value` 均为字符串。

> **重要规则（来自 IviumSoft DLL 参考手册）：**
> - 参数名**区分大小写**，必须与 IviumSoft 方法标签页中显示的字段标签完全匹配。
> - 只能更改**顶层可见参数** — 隐藏在复选框后或弹窗中的参数不可访问。
> - 更改方法类型时，先设置 `'Method'`，再设置 `'Technique'`。顺序错误会被静默忽略。
> - 如果 DLL 拒绝某参数（返回码 1 或 2），Pyvium 会抛出 `IllegalCommandError` 或 `ValueError`。当参数名或值不确定时，请用 `try/except` 包裹调用。

### 更改方法类型

```python
# 始终先设置 Method，再设置 Technique
Pyvium.set_method_parameter('Method',    'CyclicVoltammetry')
Pyvium.set_method_parameter('Technique', 'Standard')
```

### 常见扫描参数（名称必须与 IviumSoft UI 完全匹配）

| 参数名 | 典型方法 | 示例值 |
|----------------|---------------|---------------|
| `'Method'` | 任意 | `'CyclicVoltammetry'`、`'LinearSweep'`、`'Potentiostatic'` |
| `'Technique'` | 任意 | `'Standard'`、`'SampledCurrent'` |
| `'E begin'` | LSV、CV | `'0.0'` |
| `'E end'` | LSV | `'1.0'` |
| `'E step'` | LSV | `'0.005'` |
| `'scanrate'` | CV | `'0.05'` |
| `'duration'` | 计时安培法 | `'10'` |
| `'freq high'` | EIS | `'100000'` |
| `'freq low'` | EIS | `'0.1'` |

> 确切名称取决于已加载的方法。请直接在 IviumSoft 中打开方法并查看字段标签以确认拼写。

In [ ]:
Pyvium.connect_device() # set_method_parameter 需要先连接设备
# 更改方法类型 — 始终先设置 Method 再设置 Technique
Pyvium.set_method_parameter('Method',    'CyclicVoltammetry')
Pyvium.set_method_parameter('Technique', 'Standard')

# 设置标量参数
Pyvium.set_method_parameter("E begin", "0.0")
Pyvium.set_method_parameter("E end",   "0.8")
Pyvium.set_method_parameter("E step",  "0.005")

# 布尔参数使用小写字符串 'true' 或 'false'
Pyvium.set_method_parameter("OCP", "true")      # 启用 OCP 预测量

# 子参数：使用点号表示法  →  "parent.child"
# 示例：设置多级计时电位法的第 1 级
Pyvium.set_method_parameter("I_levels.I[1]", "0.001")   # 1 mA（始终使用 SI 单位）
Pyvium.set_method_parameter("I_levels.I[3]", "-0.001")  # -1 mA

print("参数已更新")

## 3. 保存修改后的方法

修改参数后，可将方法保存回磁盘。这对于从 Python 构建参数化方法模板库很有用。

In [ ]:
SAVE_METHOD_PATH = r"C:/IviumStat/datafiles/TEST1_modified.imf"

Pyvium.save_method(SAVE_METHOD_PATH)
print(f"方法已保存到: {SAVE_METHOD_PATH}")

## 4. 预测量序列

在实际测量数据开始前，方法可以自动运行最多四个预测量阶段：

| 阶段 | 操作内容 |
|-------|-------------|
| 1. 吹扫 | 在电池吹扫期间等待（例如氮气吹扫） |
| 2. 预处理 | 对电极施加条件处理电位/电流 |
| 3. OCP 测量 | 测量开路电位并用作参考 |
| 4. 平衡 | 在初始电位保持直到电池稳定 |

> **进度监控的重要说明：** 在所有预测量阶段期间，即使方法正在运行，`get_available_data_points_number()` 也返回 `0`。你的轮询循环会在整个阶段持续显示 `0 个点`，直到第一个真实数据点出现。请设置适当的超时时间，而不是假设 `0` 意味着「未开始」。

> **继续按钮：** 在 IviumSoft 手动 UI 中，在预测量序列期间点击**继续**可立即跳转到实际测量（绕过剩余的预阶段）。通过 Pyvium 控制时没有 Python 等效操作 — 除非使用 `abort_method()` 中止整个方法，否则各阶段始终运行至完成。

In [ ]:
# 选项 A：启动当前已加载的方法
Pyvium.start_method()
print("方法已启动")

# 选项 B：一步加载并启动
# Pyvium.start_method(r"C:/IviumStat/AppMethods/LSV.imf")

## 5. 监控进度

`get_available_data_points_number()` 返回迄今为止已记录的数据点数量。在循环中轮询以跟踪进度并实时读取数据。

**来自手册的三个重要注意事项：**

1. **预测量阶段：** 在吹扫、预处理、OCP 和平衡期间，计数器保持为 `0`。添加总体超时以防止方法无法产生数据时循环无限挂起。

2. **HiSpeed 模式：** 当采样间隔 < 2 ms 时，IviumSoft 自动进入 HiSpeed 模式。在此模式下，数据存储在恒电位仪的内部缓冲区（最多 **8192 个点**，CV 为 32768），仅在测量完成后传输到 PC。整个过程中计数器保持 `0`，然后一步跳到最终计数。以下轮询循环对两种情况均有效 — 只需设置充裕的总体超时。

3. **混合模式（动态阈值）：** 对于瞬态方法（例如扫速很快的 CV），IviumSoft 可能在测量过程中根据实际点速率动态切换 HiSpeed 和普通流式模式。发生这种情况时，计数器会分步递增而不是一次性跳变。以下轮询策略自动处理此情况。

以下轮询循环适用于所有三种情况。

In [ ]:
TOTAL_EXPECTED_POINTS = 160  # 设置为与你方法预期点数一致
POLL_INTERVAL_S       = 0.1
OVERALL_TIMEOUT_S     = 300  # 覆盖预测量阶段 + 实际测量

t_start = time.time()
Pyvium.start_method()
while True:
    n_points = Pyvium.get_available_data_points_number()
    elapsed  = time.time() - t_start
    progress = min(100.0, 100.0 * n_points / TOTAL_EXPECTED_POINTS)
    print(f"\r进度: {n_points} 个点 ({progress:.0f}%)  [{elapsed:.0f}s]",
          end="", flush=True)

    if n_points >= TOTAL_EXPECTED_POINTS:
        break
    if elapsed > OVERALL_TIMEOUT_S:
        print("\n超时 — 正在中止")
        Pyvium.abort_method()
        break
    time.sleep(POLL_INTERVAL_S)

print("\n完成")

## 6. 读取数据点

`get_data_point(index)` 返回三个值，其含义取决于方法类型：

| 方法类型 | 值 1 | 值 2 | 值 3 |
|-------------|---------|---------|--------|
| LSV / CV | 电位 (V) | 电流 (A) | 0 |
| 计时安培法 | 时间 (s) | 电流 (A) | 电位 (V) |
| EIS | Re(Z) (Ω) | Im(Z) (Ω) | 频率 (Hz) |
| 计时电位法 | 时间 (s) | 电位 (V) | 电流 (A) |

索引从 1 开始。

In [ ]:
# 读取所有可用数据点
total_points = Pyvium.get_available_data_points_number()

data = []
for i in range(1, total_points + 1):
    v1, v2, v3 = Pyvium.get_data_point(i)
    data.append([v1, v2, v3])

# 构建 DataFrame — 列名取决于你的方法类型
df = pd.DataFrame(data, columns=["电位 (V)", "电流 (A)", "_unused"])
print(f"已读取 {len(df)} 个数据点")
print(df.head())

## 7. 绘制结果 — LSV 示例

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["电位 (V)"], df["电流 (A)"] * 1e6, 'b-', lw=1.5)
ax.set_xlabel("电位 (V)")
ax.set_ylabel("电流 (µA)")
ax.set_title("线性扫描伏安法")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. EIS 示例 — Nyquist 图

In [ ]:
# 加载并运行 EIS 方法，然后获取数据
EIS_METHOD = r"C:/IviumStat/datafiles/EIS.imf" # 请更新此路径

# Pyvium.start_method(EIS_METHOD)
# ... 等待完成 ...

total = Pyvium.get_available_data_points_number()
eis_data = []
for i in range(1, total + 1):
    Z_re, Z_im, freq = Pyvium.get_data_point(i)
    eis_data.append([freq, Z_re, Z_im])

eis_df = pd.DataFrame(eis_data, columns=["频率 (Hz)", "Z_re (Ω)", "Z_im (Ω)"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(eis_df["Z_re (Ω)"], -eis_df["Z_im (Ω)"], 'ro-', markersize=4)
ax.set_xlabel("Z' (Ω)")
ax.set_ylabel("-Z'' (Ω)")
ax.set_title("Nyquist 图")
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 读取多扫描数据（CV）

对于包含多次扫描的循环伏安法（CV），`get_data_point_from_scan(point_index, scan_index)` 允许你从任意一次扫描获取数据 — 不仅限于最后一次。

`point_index` 和 `scan_index` 均从 1 开始。

In [ ]:
CV_METHOD       = r"C:/IviumStat/datafiles/CV.imf" # 请更新此路径
N_SCANS         = 3
POINTS_PER_SCAN = 200  # 更新为与你方法一致

# Pyvium.start_method(CV_METHOD)
# ... 等待所有扫描完成 ...

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis([i / N_SCANS for i in range(N_SCANS)])

for scan in range(1, N_SCANS + 1):
    potentials, currents = [], []
    for pt in range(1, POINTS_PER_SCAN + 1):
        E, I, _ = Pyvium.get_data_point_from_scan(pt, scan)
        potentials.append(E)
        currents.append(I * 1e6)  # µA
    ax.plot(potentials, currents, color=colors[scan - 1], label=f"扫描 {scan}")

ax.set_xlabel("电位 (V)")
ax.set_ylabel("电流 (µA)")
ax.set_title(f"循环伏安法 — {N_SCANS} 次扫描")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. 中止运行中的方法

`abort_method()` **不是即时的**。根据手册：测量在下一个数据点处停止。在采样间隔很长的情况下（例如慢速 EIS 扫描中每点间隔 60 s），中止可能需要等待完整的剩余间隔才生效。此外，仪器将缓冲数据刷回 PC 时可能有短暂延迟。

> **`IllegalCommandError`：** 如果 DLL 返回码 1（无活跃测量可中止），Pyvium 会抛出 `IllegalCommandError`。在超时循环中防御性地调用 `abort_method()` 时，捕获此异常以处理方法已自行结束的情况：
> ```python
> try:
>     Pyvium.abort_method()
> except IllegalCommandError:
>     pass  # 没有正在运行的内容 — 这没问题
> ```

In [ ]:
# 在方法运行时调用此函数可立即停止
# Pyvium.abort_method()
# print("方法已中止")

# 示例：如果测量耗时过长则中止
TIMEOUT_S = 60

# Pyvium.start_method()
t_start = time.time()
while Pyvium.get_available_data_points_number() < TOTAL_EXPECTED_POINTS:
    if time.time() - t_start > TIMEOUT_S:
        Pyvium.abort_method()
        print("已达到超时 — 方法已中止")
        break
    time.sleep(0.5)
else:
    print("方法在超时前完成")

## 11. 保存结果

`save_data()` 将最后一次测量保存到用户指定的 IDF 文件。`save_dataset()` 保存当前测量列表中的所有测量数据。

**自动保存：** 方法完成后，IviumSoft 会自动将数据保存到其默认位置。`save_data()` 是对你控制的路径的额外保存 — 不会替代自动保存，且必须在数据仍在内存中时调用（下一个方法开始前）。

> **警告：** 路径无效时调用 `save_data()` 将**关闭 IviumSoft**。调用前始终使用绝对路径并验证目录是否存在。

In [ ]:
import os

OUTPUT_DIR  = r"C:/IviumStat/Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "my_measurement.idf")

if os.path.isdir(OUTPUT_DIR):
    Pyvium.save_data(OUTPUT_FILE)
    print(f"数据已保存到: {OUTPUT_FILE}")
else:
    print(f"输出目录不存在: {OUTPUT_DIR}")
    print("请先创建该目录或更新 OUTPUT_DIR")

In [ ]:
# 将测量列表中的所有测量保存到数据集文件
DATASET_FILE = os.path.join(OUTPUT_DIR, "my_dataset.idf")

if os.path.isdir(OUTPUT_DIR):
    Pyvium.save_dataset(DATASET_FILE)
    print(f"数据集已保存到: {DATASET_FILE}")

## 12. 温度记录（Beta）

在多通道设置中，`update_temperature()` 向所有通道广播一个温度值。这是一个元数据字段 — 不控制恒温器。

In [ ]:
TEMPERATURE_CELSIUS = 25.0

Pyvium.update_temperature(TEMPERATURE_CELSIUS)
print(f"温度设置为 {TEMPERATURE_CELSIUS} °C")

## 13. SQL 数据库文件

IviumSoft 可将测量数据存储在 SQL 数据库中。`get_db_file_name()` 返回最后创建的数据库文件路径。

In [ ]:
db_path = Pyvium.get_db_file_name()
print(f"SQL 数据库: {db_path}")

## 14. 完整实验工作流

综合以上内容：一个处理错误、监控进度、读取数据并保存结果的健壮模板。

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

METHOD_PATH      = r"C:/IviumStat/datafiles/TEST1.imf"
OUTPUT_PATH      = r"C:/IviumStat/Data/TEST1_result.idf"
EXPECTED_POINTS  = 160
POLL_INTERVAL_S  = 0.2

def run_experiment(method_path, output_path, expected_points):
    print(f"启动: {os.path.basename(method_path)}")
    Pyvium.start_method(method_path)

    while True:
        n = Pyvium.get_available_data_points_number()
        print(f"\r  {n}/{expected_points} 个点", end="", flush=True)
        if n >= expected_points:
            break
        time.sleep(POLL_INTERVAL_S)
    print("  — 完成")

    rows = []
    for i in range(1, n + 1):
        rows.append(Pyvium.get_data_point(i))
    df = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])

    if os.path.isdir(os.path.dirname(output_path)):
        Pyvium.save_data(output_path)
        print(f"  已保存 → {output_path}")

    return df

Pyvium.open_driver()
Pyvium.connect_device()

df = run_experiment(METHOD_PATH, OUTPUT_PATH, EXPECTED_POINTS)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["E (V)"], df["I (A)"] * 1e6, lw=1.5)
ax.set_xlabel("电位 (V)")
ax.set_ylabel("电流 (µA)")
ax.set_title("LSV 结果")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Pyvium.close_driver()

---

## 小结

| 任务 | 方法 |
|------|--------|
| 加载方法文件 | `load_method(path)` |
| 修改参数 | `set_method_parameter(name, value)` |
| 保存修改后的方法 | `save_method(path)` |
| 启动方法（已加载） | `start_method()` |
| 启动方法（从文件） | `start_method(path)` |
| 中止运行中的方法 | `abort_method()` |
| 检查进度 | `get_available_data_points_number()` |
| 读取数据点 | `get_data_point(index)` → (v1, v2, v3) |
| 从特定扫描读取 | `get_data_point_from_scan(pt, scan)` |
| 保存最后结果 | `save_data(path)` |
| 保存完整数据集 | `save_dataset(path)` |
| 记录温度 | `update_temperature(°C)` |
| 获取 SQL 数据库路径 | `get_db_file_name()` |

## 下一步

- **`07_data_processing.ipynb`** — 离线解析已保存的 IDF 文件（无需硬件）
- **`08_batch_and_synchronization.ipynb`** — 同时协调多台设备